In [1]:
import numpy as np
import torch
import xarray as xr
from pathlib import Path
from scipy.interpolate import griddata
from torch.utils.data import Dataset


class NordicSeaDataset(Dataset):
    """Simple PyTorch dataset for Nordic Sea fields stored in xarray datasets (NetCDF (*.nc)).
    Dataset provides an interface for accessing data.
    Inputs: Ocean variables: u,v,ssh. Forcing: u, v, slp. Static: bathymetry.
    This version returns tensors shaped [lag, channels, lat, lon] for inputs and
    [channels, lat, lon] for targets.
    """

    def __init__(
        self,
        root_dir,
        input_vars=("ssh", "u", "v", "wind_u", "wind_v", "slp"),
        target_vars=("ssh", "u", "v"),
        bathymetry_var="bathymetry",
        input_lag=1,
        time_dim=None,
        temporal_window=1,
        return_temporal=False,
        transform=None,
        start_time=None,
        end_time=None,
    ):
        """Initialise the dataset with paths to the NetCDF inputs."""
        self.root = Path(root_dir).expanduser()
        if not self.root.is_absolute():
            self.root = (Path.cwd() / self.root).resolve()
        if not self.root.exists():
            fallback_root = (Path.cwd() / "data").resolve()
            self.root = fallback_root if fallback_root.exists() else self.root

        self.transform = transform
        self.input_vars = list(input_vars)
        self.target_vars = list(target_vars)

        self.input_lag = int(input_lag)
        self.temporal_window = int(temporal_window)
        self.return_temporal = bool(return_temporal)
        self.start_time = start_time
        self.end_time = end_time

        self.ssh_ds = xr.open_mfdataset(
            str(self.root / "*ssh.nc"),
            combine="by_coords",
            chunks={"time_counter": 10},
            engine="netcdf4",
            lock=False,
        )["ssh"].astype("float32")
        self.u_ds = xr.open_mfdataset(
            str(self.root / "*ubar.nc"),
            combine="by_coords",
            chunks={"time_counter": 10},
            engine="netcdf4",
            lock=False,
        )["ubar"].astype("float32")
        self.v_ds = xr.open_mfdataset(
            str(self.root / "*vbar.nc"),
            combine="by_coords",
            chunks={"time_counter": 10},
            engine="netcdf4",
            lock=False,
        )["vbar"].astype("float32")
        self.wind_u_ds = xr.open_mfdataset(
            str(self.root / "fno_ERA5forcing*.nc"),
            combine="by_coords",
            chunks={"time": 10},
            engine="netcdf4",
            lock=False,
        )["u10"].astype("float32")
        self.wind_v_ds = xr.open_mfdataset(
            str(self.root / "fno_ERA5forcing*.nc"),
            combine="by_coords",
            chunks={"time": 10},
            engine="netcdf4",
            lock=False,
        )["v10"].astype("float32")
        self.slp_ds = xr.open_mfdataset(
            str(self.root / "fno_ERA5forcing*.nc"),
            combine="by_coords",
            chunks={"time": 10},
            engine="netcdf4",
            lock=False,
        )["msl"].astype("float32")

        self.ssh_ds = self.ssh_ds.rename({"time_counter": "time"})
        self.u_ds = self.u_ds.rename({"time_counter": "time"})
        self.v_ds = self.v_ds.rename({"time_counter": "time"})

        self.wind_u_ds = self.wind_u_ds.astype("float32")
        self.wind_v_ds = self.wind_v_ds.astype("float32")
        self.slp_ds = self.slp_ds.astype("float32")

        self.bathy_ds = xr.open_dataset(str(self.root / "nordic_seas_domain_cfg.nc"), engine="netcdf4")["bathy_metry"]

        self.var_map = {
            "ssh": self.ssh_ds,
            "u": self.u_ds,
            "v": self.v_ds,
            "wind_u": self.wind_u_ds,
            "wind_v": self.wind_v_ds,
            "slp": self.slp_ds,
        }

        if self.start_time is not None or self.end_time is not None:
            self.ssh_ds = self.ssh_ds.sel(time=slice(self.start_time, self.end_time))
            self.u_ds = self.u_ds.sel(time=slice(self.start_time, self.end_time))
            self.v_ds = self.v_ds.sel(time=slice(self.start_time, self.end_time))
            self.wind_u_ds = self.wind_u_ds.sel(time=slice(self.start_time, self.end_time))
            self.wind_v_ds = self.wind_v_ds.sel(time=slice(self.start_time, self.end_time))
            self.slp_ds = self.slp_ds.sel(time=slice(self.start_time, self.end_time))
            self.var_map = {
                "ssh": self.ssh_ds,
                "u": self.u_ds,
                "v": self.v_ds,
                "wind_u": self.wind_u_ds,
                "wind_v": self.wind_v_ds,
                "slp": self.slp_ds,
            }

        self.times = self.ssh_ds.time.values
        self.n_samples = max(len(self.times) - self.input_lag, 0)
        self._source_points = None
        self._target_points = None

    def _regrid_to_ocean(self, da):
        if self._source_points is None:
            forcing_lat = self.wind_u_ds.lat.values
            forcing_lon = self.wind_u_ds.lon.values
            forcing_lon2d, forcing_lat2d = np.meshgrid(forcing_lon, forcing_lat)
            self._source_points = np.column_stack((forcing_lat2d.ravel(), forcing_lon2d.ravel()))

        if self._target_points is None:
            ocean_lat = self.ssh_ds.nav_lat.values
            ocean_lon = self.ssh_ds.nav_lon.values
            self._target_points = np.column_stack((ocean_lat.ravel(), ocean_lon.ravel()))

        values = np.asarray(da.values, dtype=np.float32).ravel()
        regridded = griddata(
            self._source_points,
            values,
            self._target_points,
            method="linear",
            fill_value=np.nan,
        )
        if np.isnan(regridded).any():
            nearest = griddata(
                self._source_points,
                values,
                self._target_points,
                method="nearest",
            )
            regridded[np.isnan(regridded)] = nearest[np.isnan(regridded)]

        target_shape = (self.ssh_ds.sizes.get("y"), self.ssh_ds.sizes.get("x"))
        return regridded.reshape(target_shape).astype(np.float32)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        input_times = self.times[idx : idx + self.input_lag]
        target_time = self.times[idx + self.input_lag]

        input_frames = []
        for t in input_times:
            channels = []
            for var in self.input_vars:
                da = self.var_map.get(var)
                if da is None:
                    raise KeyError(f"Input variable {var!r} not recognised")
                if var in {"wind_u", "wind_v", "slp"}:
                    field = da.sel(time=t, method="nearest")
                    channels.append(self._regrid_to_ocean(field))
                else:
                    channels.append(da.sel(time=t, method="nearest").astype(np.float32).values)
            frame = np.stack(channels, axis=0)
            input_frames.append(frame)

        x = np.stack(input_frames, axis=0)

        target_channels = []
        for var in self.target_vars:
            da = self.var_map.get(var)
            if da is None:
                raise KeyError(f"Target variable {var!r} not recognised")
            if var in {"wind_u", "wind_v", "slp"}:
                field = da.sel(time=target_time, method="nearest")
                target_channels.append(self._regrid_to_ocean(field))
            else:
                target_channels.append(da.sel(time=target_time, method="nearest").astype(np.float32).values)

        y = np.stack(target_channels, axis=0)

        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        if self.transform is not None:
            x, y = self.transform(x, y)

        return x, y

In [4]:
from utils.DataLoader import NordicSeaDataset

train_dataset = NordicSeaDataset(
    root_dir="./data",
    start_time="1980-01-01",
    end_time="2009-12-31 23:00",
    input_lag=1,
    coarse_step=5,
    lat_trim=50,
    lon_trim=0,
)
print(train_dataset)
print("start_time:", train_dataset.start_time)
print("end_time:", train_dataset.end_time)
print("dataset time range:", train_dataset.times[0], "to", train_dataset.times[-1])
print("n_samples:", len(train_dataset))
print("input_lag:", train_dataset.input_lag)
print("input shape:", train_dataset[0][0].shape)
print("target shape:", train_dataset[0][1].shape)

start_time: 1980-01-01
end_time: 2009-12-31 23:00
dataset time range: 1980-01-01T00:30:00.000000000 to 1980-12-31T23:30:00.000000000
n_samples: 8783
input_lag: 1
input shape: torch.Size([1, 7, 46, 46])
target shape: torch.Size([3, 280, 230])


In [3]:
train_dataset = NordicSeaDataset(
    root_dir="../data",
    start_time="1980-01-01",
    end_time="2009-12-31 23:00",
    input_lag=4
)